# Phase 2: Micro-Level Data Preprocessing

This notebook transforms raw job postings scraped from Jobup.ch into a clean, analysis-ready micro-level dataset for integration with BFS macro-level vacancy data.

**Input:** `jobs_combined_2026-03-01T17-21-57.csv` — 743 job postings across 5 roles (Data Scientist, Data Analyst, ML Engineer, Data Engineer, AI Engineer)  
**Output:** `jobs_micro_cleaned_final.csv` — 743 rows × 20 columns

---

## Table of Contents
1. [Load Dataset & Structural Validation](#step1)
2. [Structural Cleaning](#step2)
3. [Column Standardization](#step3)
4. [Quarter Variable Construction](#step4)
5. [Skill Extraction](#step5)
6. [Salary Transparency Reconstruction](#step6)
7. [Seniority Classification](#step7)
8. [City, Canton & Region Mapping](#step8)
9. [Workload Cleaning](#step9)
10. [Final Column Summary](#step10)
11. [Save Final Dataset](#step11)

## Step 1: Load Dataset & Structural Validation

The raw scraped dataset is loaded from the processed data folder and inspected for basic structural integrity. This includes verifying the number of observations, confirming that all expected columns are present, and checking for missing values before any transformations are applied.

In [1]:
import pandas as pd
import numpy as np
import re

# Load dataset
df = pd.read_csv("../data/processed/jobs_combined_2026-03-01T17-21-57.csv")

print("Shape:", df.shape)
df.head()

Shape: (743, 19)


,source,url,job_id,search_term,title,company,location_raw,posted_date,employment_type,workload_raw,salary_raw,description_raw,scraped_at,canton,seniority,description_clean,skills,skill_count,salary_available
0,jobup,https://www.jobup.ch/en/jobs/detail/a5de724a-4...,43725c60f2a0e7a20223831427ec1c5733c8e34c,data scientist,Mission étudiante 40% : Data Analyst,Academic Work Switzerland,Montreux,20 February 2026,Temporary,40 – 50%,NaN,About the job Tu es actuellement étudiant.e ? ...,2026-03-01T17:21:57,NaN,NaN,About the job Tu es actuellement étudiant.e ? ...,NaN,0,0
1,jobup,https://www.jobup.ch/en/jobs/detail/366613c5-e...,929b4733c6e1c66d045413a3373023e435e25a0c,data scientist,Junior Survey Data Analyst 20%,Academic Work Switzerland,Lausanne,10 February 2026,Temporary,20%,NaN,About the job Tu es étudiant.e et recherches u...,2026-03-01T17:21:57,NaN,NaN,About the job Tu es étudiant.e et recherches u...,NaN,0,0
2,jobup,https://www.jobup.ch/en/jobs/detail/99ded019-3...,da46ba822882645bc160031dd473b01555e8e9c8,data scientist,Merchandising & Data Analyst (H/F),Vilebrequin,Plan-les-Ouates,06 February 2026,Temporary,60 – 100%,NaN,"About the job Née en 1971 à St-Tropez, la mais...",2026-03-01T17:21:57,NaN,NaN,"About the job Née en 1971 à St-Tropez, la mais...",NaN,0,0
3,jobup,https://www.jobup.ch/en/jobs/detail/6f302ed9-c...,face52ba0e66311274a47f6df9fbf699abc6bc3d,data scientist,Spécialiste informatique - Interfaces et donné...,Transports publics fribourgeois (TPF) SA,Givisiez,01 March 2026,Permanent position,80 – 100%,NaN,About the job La mobilité est la raison d'être...,2026-03-01T17:21:57,NaN,NaN,About the job La mobilité est la raison d'être...,NaN,0,0
4,jobup,https://www.jobup.ch/en/jobs/detail/fb39e340-5...,0e5906d2b6906c6f4eac077cdb6cda27d9955253,data scientist,Data Scientist - Innovation Collaborations,EPFL,Lausanne,20 February 2026,Temporary,100%,NaN,"About the job EPFL, the Swiss Federal Institut...",2026-03-01T17:21:57,NaN,NaN,"About the job EPFL, the Swiss Federal Institut...",NaN,0,0


## Step 2: Structural Cleaning

Duplicate job postings are removed based on `job_id`, which serves as the unique identifier for each scraped listing. The index is reset after deduplication to maintain a clean sequential structure for downstream processing.

In [2]:
# Basic Structural Cleaning

# Remove duplicates
df = df.drop_duplicates(subset="job_id")

# Reset index
df = df.reset_index(drop=True)

print("Remaining jobs:", len(df))

Remaining jobs: 743


## Step 3: Column Standardization & Naming Conventions

Column names are renamed to match the project's final schema. `search_term` becomes `role` and `description_clean` becomes `description`. The `role` field is also normalised to lowercase to ensure consistent matching across the pipeline.

In [3]:
# Column Standardisation — rename to final schema and normalise role labels

df = df.rename(columns={
    "search_term": "role",
    "description_clean": "description"
})

df["role"] = df["role"].str.lower().str.strip()

print("Columns:", df.columns.tolist())

Columns: ['source', 'url', 'job_id', 'role', 'title', 'company', 'location_raw', 'posted_date', 'employment_type', 'workload_raw', 'salary_raw', 'description_raw', 'scraped_at', 'canton', 'seniority', 'description', 'skills', 'skill_count', 'salary_available']


## Step 4: Quarter Variable Construction

`posted_date` is converted to datetime format and used to derive a `quarter` variable (e.g. `2026Q1`) for time-based analysis. A `macro_quarter` column is initialised as a copy of `quarter` and will be used in Phase 3 to align job postings with the BFS vacancy release window.

In [4]:
df["posted_date"] = pd.to_datetime(df["posted_date"], errors="coerce")

df["quarter"] = df["posted_date"].dt.to_period("Q").astype(str)

# macro_quarter will later be used for macro merge
df["macro_quarter"] = df["quarter"]

## Step 5: Skill Extraction

Skills are extracted from the job title, description, and role fields using a predefined dictionary of 100+ technologies across programming languages, ML frameworks, cloud platforms, data engineering tools, and more. All patterns use `\b` word boundaries to prevent false positives from substrings. For example `r` matching *recherche*, `git` matching *digital*, or `rag` matching *paragraphe* in French-language postings.

In [5]:
# Skill Dictionary (Focus: Data Engineering / DS / ML / AI)

skill_patterns = [

# programming
"python", "r", "sql", "scala", "java", "c++", "bash", "rust",

# python ecosystem
"pandas", "numpy", "scipy", "scikit-learn", "sklearn", "polars",
"jupyter", "pydantic", "fastapi",

# machine learning
"machine learning", "deep learning", "artificial intelligence",
"tensorflow", "keras", "pytorch", "xgboost", "lightgbm", "catboost",
"random forest", "gradient boosting", "feature engineering",
"transfer learning", "reinforcement learning",

# generative AI & LLMs (NEW - very high demand)
"llm", "large language model", "generative ai", "gen ai",
"gpt", "claude", "gemini", "llama", "mistral",
"prompt engineering", "rag", "retrieval augmented generation",
"langchain", "llamaindex", "hugging face", "transformers",
"fine-tuning", "lora", "qlora",
"ai agents", "agentic", "autonomous agents",
"vector database", "vector store", "embeddings",
"pinecone", "weaviate", "chroma", "qdrant",

# NLP
"nlp", "natural language processing",
"bert", "t5", "text classification", "named entity recognition",
"ner", "sentiment analysis", "semantic search",

# computer vision
"computer vision", "image recognition", "object detection",
"opencv", "yolo", "diffusion models", "multimodal",

# statistics
"statistics", "regression", "classification", "clustering",
"time series", "forecasting", "hypothesis testing", "a/b testing",
"bayesian",

# mlops (growing fast - appeared in ~8% of postings, up sharply)
"mlops", "ml ops", "model deployment", "model serving",
"model monitoring", "model drift", "ci/cd",
"mlflow", "weights & biases", "wandb", "bentoml",
"seldon", "kubeflow", "sagemaker", "vertex ai",

# data engineering (VERY IMPORTANT)
"etl", "elt", "data pipeline", "data pipelines", "data engineering",
"spark", "pyspark", "hadoop", "kafka", "flink",
"dbt", "data build tool", "data modeling",
"data lake", "data lakehouse", "data mesh",
"delta lake", "apache iceberg", "apache hudi",

# orchestration
"airflow", "luigi", "prefect", "dagster",

# data warehouses
"snowflake", "bigquery", "redshift", "databricks",

# databases
"postgresql", "mysql", "mongodb", "redis",
"elasticsearch", "opensearch", "neo4j",

# cloud
"aws", "gcp", "google cloud", "azure",
"cloud computing",

# infrastructure
"docker", "kubernetes", "terraform", "helm",
"serverless", "lambda", "cloud functions",

# dev tools
"git", "github", "gitlab", "linux",
"rest api", "graphql", "microservices",

# analytics
"data analysis", "data analytics", "business intelligence",
"product analytics", "growth analytics",

# visualization
"tableau", "power bi", "looker", "matplotlib",
"seaborn", "plotly", "d3.js", "grafana",

# spreadsheets
"excel",

# responsible / enterprise AI (emerging)
"responsible ai", "ai governance", "model explainability",
"fairness", "bias detection", "privacy", "gdpr",
"feature store",

# German equivalents
"ki",
"maschinelles lernen",
"datenwissenschaft",
"datenanalyse",
"visualisierung",
]

In [6]:
# Skill Extraction
# Uses \b word boundary for all skills — prevents false positives from
# substrings like: scala→scalable, rust→trust, git→digital, excel→excellent,
# rag→paragraphe (French), r→résultats (French)

import re

def extract_skills(row):
    text = " ".join([
        str(row["title"]),
        str(row["description"]),
        str(row["role"])
    ]).lower()

    found = []
    for skill in skill_patterns:
        pattern = r'\b' + re.escape(skill.lower()) + r'\b'
        if re.search(pattern, text):
            found.append(skill)

    return sorted(list(set(found)))

In [7]:
# Apply skill extraction
df["skills"] = df.apply(extract_skills, axis=1)
df["skill_count"] = df["skills"].apply(len)

In [8]:
# Check skill distribution
df["skill_count"].describe()

count    743.000000
mean       2.823688
std        4.255313
min        0.000000
25%        0.000000
50%        1.000000
75%        4.000000
max       46.000000
Name: skill_count, dtype: float64

In [9]:
# Check how many jobs still have 0 skills
(df["skill_count"] == 0).sum()

283

In [10]:
# Check how often R appears
sum("r" in skills for skills in df["skills"])

69

In [11]:
# Last Check

df[["role","title","skills"]].sample(10, random_state=42)

,role,title,skills
609,data engineer,Projektportfolio Agile Flow Lead (w/m/d),[]
539,data engineer,Electrical Engineer in Product Care 80 – 100 %...,[]
693,data engineer,Ingenieur Maschinenbau / Verfahrenstechnik - T...,[]
350,data engineer,Process Engineer Oats,"[excel, r]"
174,data analyst,IT Business Analyst & Project Manager,[]
81,data scientist,Senior Scientist/Coordinator for high throughp...,[]
355,data engineer,Data Ingestion Engineer (all),"[azure, ci/cd, data engineering, dbt, git, pyt..."
424,data engineer,Principal/Staff Software Engineer,"[ai agents, aws, ci/cd, fine-tuning, flink, gd..."
523,data engineer,ICT System Engineer – Datacenter & Workplace 8...,[azure]
617,data engineer,Senior Azure Spezialist,"[azure, ci/cd, gitlab, terraform]"


## Step 6: Salary Transparency Reconstruction

A binary indicator `salary_available` is created from the raw salary field (1 = salary disclosed, 0 = not disclosed). Where salary information is present, `salary_min` and `salary_max` are extracted from the description text using pattern matching across CHF formats in German, French, and English. Given the very low disclosure rate in Swiss job postings, `salary_available` is the primary analytical variable, salary levels themselves are too sparse for reliable comparison.

In [12]:
# Create binary indicator for salary transparency
df["salary_available"] = df["salary_raw"].notna().astype(int)

# Quick check
df["salary_available"].value_counts(dropna=False)

salary_available
0    743
Name: count, dtype: int64

In [13]:
def extract_salary(desc):
    if pd.isna(desc) or 'chf' not in desc.lower():
        return None, None
    
    # Normalize unicode apostrophes
    desc = desc.replace('\u2019', "'").replace('\u2018', "'").replace('\u2032', "'")
    
    # Pattern 1: CHF 110'000 to/bis/- 130'000
    for m in re.finditer(r"chf\s*([\d']+)\s*(?:to|bis|-|–)\s*([\d']+)", desc, re.IGNORECASE):
        lo = int(re.sub(r"[',]", "", m.group(1)))
        hi = int(re.sub(r"[',]", "", m.group(2)))
        if 10000 <= lo <= 500000 and 10000 <= hi <= 500000:
            return lo, hi
    
    # Pattern 2: CHF 110'000 single value
    for m in re.finditer(r"chf\s*([\d']['\d]+)", desc, re.IGNORECASE):
        val = int(re.sub(r"[',]", "", m.group(1)))
        if 10000 <= val <= 500000:
            return val, val
    
    # Pattern 3: 3000 CHF per month → annualize
    for m in re.finditer(r"(\d+)\s*chf\s*per\s*month", desc, re.IGNORECASE):
        val = int(m.group(1)) * 12
        if 10000 <= val <= 500000:
            return val, val

    # Pattern 4: CHF 88,000
    for m in re.finditer(r"chf\s*([\d,]+)", desc, re.IGNORECASE):
        val = int(re.sub(r",", "", m.group(1)))
        if 10000 <= val <= 500000:
            return val, val

    return None, None

df["salary_min"], df["salary_max"] = zip(*df["description"].apply(extract_salary))
df["salary_available"] = df["salary_min"].notna().astype(int)

## Step 7: Seniority Classification

Seniority is inferred in two passes. First, job titles are scanned for explicit signals using `\b`-bounded regex (e.g. `intern`, `junior`, `senior`, `lead`). Unclassified rows then fall through to a description fallback that looks for experience year mentions such as `"3-5 years of experience"`, mapped as ≤2y → junior, 3–5y → mid, 6y+ → senior. Postings with no detectable signal are retained as `None`, approximately 50% of the dataset, reflecting the common Swiss practice of omitting seniority from job titles.

In [14]:
# Seniority Classification
# Uses re.search with \b boundaries throughout — consistent with skill extraction approach.
# Title is checked first (most reliable signal), description used as conservative fallback.
# Returns None instead of defaulting to "mid" — unknown seniority stays explicit.

def classify_seniority(row):
    title = str(row["title"]).lower()
    desc  = str(row["description"]).lower()

    # ── Title-first (most reliable) ───────────────────────────────────────────
    if re.search(r'\b(intern|internship|stage|stagiaire|praktikant|werkstudent|trainee)\b', title):
        return "intern"
    if re.search(r'\b(postdoc|postdoctoral)\b', title):
        return "mid"
    if re.search(r'\b(junior|jr\.?)\b', title):
        return "junior"
    if re.search(r'\b(lead|principal|head of|chapter lead|teamleiter|manager)\b', title):
        return "lead"
    if re.search(r'\b(senior|sr\.?)\b', title):
        return "senior"
    if re.search(r'\b(mid[- ]level|medior)\b', title):
        return "mid"

    # ── Description fallback — only unambiguous signals ───────────────────────
    if re.search(r'\b(internship|praktikum|werkstudent|stipend)\b', desc):
        return "intern"
    if re.search(r'\b(postdoc|postdoctoral)\b', desc):
        return "mid"
    if re.search(r'\b(entry[- ]level|berufseinsteiger|first.?job)\b', desc):
        return "junior"
    if re.search(r'\b(you will lead|leading a team|manage a team|verantwortlich für das team)\b', desc):
        return "lead"

    return None  # genuinely no seniority signal — keep as explicit null

df["seniority"] = df.apply(classify_seniority, axis=1)

print(df["seniority"].value_counts(dropna=False))
print(f"\nUnclassified: {df['seniority'].isna().sum()} / {len(df)} ({df['seniority'].isna().mean()*100:.1f}%)")

seniority
None      472
senior    111
lead       82
intern     44
mid        20
junior     14
Name: count, dtype: int64

Unclassified: 472 / 743 (63.5%)


In [15]:
# Experience-based seniority fallback
# Applied only to rows still unclassified after title/description pass
# For ranges (e.g. "3-5 years"), uses lower bound to be conservative

exp_experience_pattern = re.compile(
    r'(\d+)\+?\s*'
    r'(?:to|-|bis|à)\s*(\d+)\+?\s*'
    r'(?:years?|ans?|jahre?|anni)'
    r'(?:\s+of)?\s*'
    r'(?:experience|expérience|erfahrung|esperienza)',
    re.IGNORECASE
)

exp_single_pattern = re.compile(
    r'(\d+)\+?\s*'
    r'(?:years?|ans?|jahre?|anni)'
    r'(?:\s+of)?\s*'
    r'(?:experience|expérience|erfahrung|esperienza)',
    re.IGNORECASE
)

def years_to_seniority(years):
    if years <= 2:  return "junior"
    elif years <= 5: return "mid"
    else:            return "senior"

def extract_seniority_from_experience(desc):
    if pd.isna(desc):
        return None
    match = exp_experience_pattern.search(desc)
    if match:
        return years_to_seniority(int(match.group(1)))
    match = exp_single_pattern.search(desc)
    if match:
        return years_to_seniority(int(match.group(1)))
    return None

mask = df["seniority"].isna()
df.loc[mask, "seniority"] = df.loc[mask, "description"].apply(extract_seniority_from_experience)

print(df["seniority"].value_counts(dropna=False))
print(f"\nNull: {df['seniority'].isna().sum()} / {len(df)} ({df['seniority'].isna().mean()*100:.1f}%)")


seniority
None      394
senior    121
lead       82
mid        72
intern     44
junior     30
Name: count, dtype: int64

Null: 394 / 743 (53.0%)


## Step 8: City, Canton, Region Mapping

Location is parsed from `location_raw` in three sub-steps. Canton is extracted using a 7-step priority chain: explicit `CH-XX` codes, known company-name lookups, canton name patterns in German/French/Italian, and a city-to-canton dictionary. City is then extracted from the first segment of the location string. Finally, cantons are mapped to the seven BFS Major Regions used in the macro-level vacancy data, enabling the Phase 3 merge.

In [16]:
# Canton Parser
# Priority order:
# 1. CH-XX code in string
# 2. Known company name lookup
# 3. Kanton name pattern (e.g. "Kanton Zürich", "Vaud")
# 4. Skip company/region labels
# 5. City → canton map

import re

canton_codes = [
    "ZH","BE","LU","UR","SZ","OW","NW","GL","ZG","FR","SO",
    "BS","BL","SH","AR","AI","SG","GR","AG","TG","TI","VD","VS","NE","GE","JU"
]

company_markers = [
    "AG", "SA", "GmbH", "Sàrl", "Sagl", "Group", "Inc", "Corp",
    "Holding", "Bank", "Institut", "Hochschule", "Université",
    "Genossenschaft", "Versicherung", "Spital", "Foundation",
    "Consulting", "Engineering", "Technologies", "Systems",
    "Services", "Solutions", "Analytics", "Media", "Partners",
    "Association", "NCCR"
]

region_junk = [
    "Deutschschweiz", "Innerschweiz", "Ostschweiz", "Mittelland",
    "Zentralschweiz", "Region ", "Raum ", "Liechtenstein"
]

kanton_name_map = {
    "zürich": "ZH", "zurich": "ZH", "bern": "BE", "berne": "BE",
    "luzern": "LU", "lucerne": "LU", "zug": "ZG", "aargau": "AG",
    "thurgau": "TG", "graubünden": "GR", "solothurn": "SO",
    "schaffhausen": "SH", "fribourg": "FR", "freiburg": "FR",
    "neuchâtel": "NE", "neuchatel": "NE", "genf": "GE", "geneva": "GE",
    "genève": "GE", "vaud": "VD", "wallis": "VS", "valais": "VS",
    "ticino": "TI", "tessin": "TI", "appenzell": "AR", "schwyz": "SZ",
    "glarus": "GL", "uri": "UR", "nidwalden": "NW", "obwalden": "OW",
    "jura": "JU", "basel-stadt": "BS", "basel-land": "BL", "baselland": "BL",
}

city_to_canton = {
    # ZH
    "Zürich": "ZH", "Zurich": "ZH", "Winterthur": "ZH", "Schlieren": "ZH",
    "Männedorf": "ZH", "Dübendorf": "ZH", "Wallisellen": "ZH", "Kloten": "ZH",
    "Opfikon": "ZH", "Uster": "ZH", "Glattpark": "ZH", "Volketswil": "ZH",
    "Dietikon": "ZH", "Thalwil": "ZH", "Stäfa": "ZH", "Rüti ZH": "ZH",
    "Wetzikon ZH": "ZH", "Buchs ZH": "ZH", "Seuzach": "ZH", "Fehraltorf": "ZH",
    "Zürcher Oberland": "ZH", "Zürich Oerlikon": "ZH", "Kanton Zürich": "ZH",
    "Rüschlikon": "ZH", "Dietlikon": "ZH",
    # BE
    "Bern": "BE", "Berne": "BE", "Biel": "BE", "Thun": "BE", "Köniz": "BE",
    "Burgdorf": "BE", "Ostermundigen": "BE", "Kehrsatz": "BE",
    "Affoltern im Emmental": "BE", "Niederönz": "BE",
    # BS / BL
    "Basel": "BS", "Basel-Stadt": "BS",
    "Allschwil": "BL", "Liestal": "BL", "Münchenstein": "BL", "Bubendorf": "BL",
    # VD
    "Lausanne": "VD", "Montreux": "VD", "Gland": "VD", "Nyon": "VD",
    "Yverdon-les-Bains": "VD", "Yverdon": "VD", "Morges": "VD",
    "Bussigny-près-Lausanne": "VD", "Epalinges": "VD", "Crissier": "VD",
    "Renens VD": "VD", "Vevey": "VD", "Corsier-sur-Vevey": "VD",
    # GE
    "Geneva": "GE", "Genève": "GE", "Geneve": "GE",
    "Plan-les-Ouates": "GE", "Meyrin": "GE", "Carouge": "GE",
    "Petit-Lancy": "GE", "Grand-Lancy": "GE",
    # VS
    "Sion": "VS", "Martigny": "VS", "Wallis": "VS", "Sierre": "VS",
    "Monthey": "VS", "Chalais": "VS",
    # FR
    "Fribourg": "FR", "Givisiez": "FR", "Freiburg": "FR", "Bulle": "FR", "Morat": "FR",
    # SG
    "St. Gallen": "SG", "Saint Gallen": "SG", "Uzwil": "SG",
    "Gossau": "SG", "Wil": "SG", "Ebnat-Kappel": "SG", "Haag": "SG",
    # TI
    "Lugano": "TI", "Bellinzona": "TI", "Losone": "TI",
    # LU
    "Lucerne": "LU", "Luzern": "LU", "Sursee": "LU", "Kriens": "LU",
    # AG
    "Aarau": "AG", "Baden": "AG", "Lenzburg": "AG", "Brugg": "AG",
    "Zofingen": "AG", "Dättwil": "AG", "Ehrendingen": "AG", "Aargau": "AG",
    # ZG
    "Zug": "ZG", "Rotkreuz": "ZG", "Baar": "ZG", "Cham": "ZG",
    # SO
    "Solothurn": "SO", "Olten": "SO",
    # NE
    "Neuchâtel": "NE", "Neuchatel": "NE", "La Chaux-de-Fonds": "NE",
    "Le Crêt-du-Locle": "NE", "Le Landeron": "NE",
    # GR
    "Chur": "GR",
    # TG
    "Frauenfeld": "TG", "Romanshorn": "TG", "Egnach": "TG",
    "Sulgen": "TG", "Münchwilen": "TG", "Kanton Thurgau": "TG",
    # SH
    "Schaffhausen": "SH",
    # SZ
    "Schwyz": "SZ", "Lachen SZ": "SZ", "Pfäffikon": "SZ",
    "Unteriberg": "SZ", "Feusisberg": "SZ", "Wollerau": "SZ",
}

company_to_canton = {
    # GE
    "CERN European Organization for Nuclear Research": "GE",
    "Banque Pictet & Cie SA": "GE",
    "Bank Julius Bär & Co. AG": "GE",
    "Banque Lombard Odier & Cie SA": "GE",
    "Vitol": "GE",
    "Kudelski S.A.": "GE",
    "H55 SA": "GE",
    "Energy Vault SA": "GE",
    "HYDRO Exploitation SA": "GE",
    "Kepler Cheuvreux (Suisse) SA": "GE",
    "SIB Institut Suisse de Bioinformatique": "GE",
    # ZH
    "Scandit AG": "ZH", "Hitachi Energy AG": "ZH", "On AG": "ZH",
    "SonarSource SA": "ZH", "Nexthink SA": "ZH", "GetYourGuide AG": "ZH",
    "SkyCell AG": "ZH", "ti&m AG": "ZH", "Avaloq Group AG": "ZH",
    "lastminute.com group": "ZH", "Daedalean AG": "ZH", "SIX Group AG": "ZH",
    "Open Systems AG": "ZH", "Exeon Analytics AG": "ZH", "AXA Versicherungen AG": "ZH",
    "ZHAW Zürcher Hochschule für Angewandte Wissenschaften": "ZH",
    "Zühlke Engineering AG": "ZH", "Swiss International Air Lines AG": "ZH",
    "Edelweiss Air AG": "ZH", "SR Technics Switzerland AG": "ZH",
    "swissQuant Group AG": "ZH", "comparis.ch AG": "ZH",
    "SMG Swiss Marketplace Group": "ZH", "AutoForm Development GmbH": "ZH",
    "Vontobel Beteiligungen AG": "ZH", "Sonova AG": "ZH", "Wingtra AG": "ZH",
    "Frontify AG": "ZH",
    "EAWAG Dübendorf, Eidg. Anstalt für Wasser-, Abwasserreinigung & Gewässerschutz": "ZH",
    "D ONE Value Creation AG": "ZH", "adesso Schweiz AG": "ZH", "myScience": "ZH",
    "ZüriPharm AG": "ZH", "Maag Pump Systems AG": "ZH", "isolutions AG": "ZH",
    "PPCmetrics AG": "ZH", "Supercomputing Systems AG": "ZH",
    "Zürcher Hochschule der Künste": "ZH", "ServiceNow": "ZH", "Deloitte": "ZH",
    "NielsenIQ": "ZH", "Octapharma AG": "ZH", "Ernst & Young AG": "ZH",
    "Swiss Life AG": "ZH", "Sika AG": "ZH", "ISS Schweiz": "ZH",
    "Impact Hub Zürich AG": "ZH",
    # BS / BL
    "Novartis AG": "BS", "Universitätsspital Basel": "BS",
    "Basler Kantonalbank": "BS", "Bank Cler AG": "BS", "Syngenta Group": "BS",
    "Huntsman Advanced Materials (Switzerland) GmbH": "BL",
    # BE
    "Hamilton Services AG": "BE", "Energie Wasser Bern": "BE",
    "Implenia Schweiz AG": "BE", "Ramboll AG": "BE", "CSL Behring AG": "BE",
    # VD
    "Université de Lausanne": "VD", "UCB-Pharma SA": "VD",
    "Magellan Partners": "VD", "Bobst Mex SA": "VD", "salesforce.com Sàrl": "VD",
    "Amaris Consulting Sàrl": "VD", "Talan SA": "VD", "VF International Sagl": "VD",
    "selected sa": "VD", "MANTA GROUP SA": "VD",
    # AG
    "Baloise Versicherung AG": "AG", "Mettler-Toledo GmbH": "AG",
    "Thermo Fisher Scientific (Schweiz) AG": "AG",
    "GF Machining Solutions Management SA": "AG", "RUAG AG": "AG",
    "Paul Scherrer Institut (PSI)": "AG",
    # SG
    "Stadler Rail Group": "SG", "St.Galler Stadtwerke": "SG",
    "Büchi Labortechnik AG": "SG",
    # LU
    "Hochschule Luzern (HSLU)": "LU",
    # SO
    "SCOR Services Switzerland AG": "SO",
    # TI
    "Agie Charmilles SA, Losone": "TI", "Officine Ghidoni SA": "TI",
    # NE
    "Comfone AG": "NE",
    # FR
    "Etat du canton de Fribourg": "FR",
    # GR (Liechtenstein → GR as closest CH canton)
    "LGT Bank AG": "GR", "Bank Frick AG": "GR", "Prismalife AG": "GR",
    "Peoplefone AG": "GR",
}


def parse_canton(location_raw):
    if pd.isna(location_raw):
        return None

    loc = str(location_raw).strip()

    # 1. Explicit CH-XX code (e.g. "Zug, CH-ZG")
    m = re.search(r'\bCH-(' + '|'.join(canton_codes) + r')\b', loc)
    if m:
        return m.group(1)

    # 2. Known company name
    if loc in company_to_canton:
        return company_to_canton[loc]

    # 3. Kanton name anywhere in string
    loc_lower = loc.lower()
    for name, code in kanton_name_map.items():
        if name in loc_lower:
            return code

    # 4. Skip company names
    if any(marker in loc for marker in company_markers):
        return None

    # 5. Skip region labels
    if any(junk in loc for junk in region_junk):
        return None

    # 6. City map — try first token before comma
    city = loc.split(",")[0].strip()
    if city in city_to_canton:
        return city_to_canton[city]

    # 7. Try full string
    if loc in city_to_canton:
        return city_to_canton[loc]

    return None


In [17]:
# Apply canton parser
df["canton"] = df["location_raw"].apply(parse_canton)

print(f"Canton null: {df['canton'].isna().sum()} / {len(df)}")
print("\nCanton distribution:")
print(df["canton"].value_counts())

Canton null: 99 / 743

Canton distribution:
canton
ZH    242
GE     87
VD     76
BE     59
AG     30
SG     26
VS     21
BS     21
LU     12
NE     11
FR      9
SO      9
ZG      8
TG      7
SZ      6
BL      5
UR      5
GR      4
TI      4
SH      1
AR      1
Name: count, dtype: int64


In [18]:
# Check remaining nulls — should all be company names or ambiguous
df[df["canton"].isna()]["location_raw"].value_counts().head(20)

location_raw
Deutschschweiz                                        4
Region Mittelland (AG / SO) / Region Freiamt/Lenzb    3
Baden und Rothenburg                                  3
Ärztekasse Genossenschaft                             3
Thoratec Switzerland GmbH                             2
Winterthur Gas & Diesel AG                            2
INP Schweiz AG                                        2
ViaSat Antenna Systems SA                             2
Dexion Services AG                                    2
Schmerikon                                            2
Innerschweiz                                          2
Starrag Vuadens SA                                    2
Acacias                                               2
Eurofins                                              2
Celerion Switzerland AG                               2
Dietziker Partner Baumanagement AG                    1
QIM Info SA                                           1
Thun (Gwatt)                       

In [19]:
# Extract city from location_raw (first token before comma, cleaned)

company_to_city = {
    # Zürich
    "Scandit AG": "Zürich",
    "Hitachi Energy AG": "Zürich",
    "On AG": "Zürich",
    "SonarSource SA": "Zürich",
    "Nexthink SA": "Zürich",
    "GetYourGuide AG": "Zürich",
    "SkyCell AG": "Zürich",
    "ti&m AG": "Zürich",
    "Avaloq Group AG": "Zürich",
    "lastminute.com group": "Zürich",
    "Daedalean AG": "Zürich",
    "SIX Group AG": "Zürich",
    "Open Systems AG": "Zürich",
    "Exeon Analytics AG": "Zürich",
    "AXA Versicherungen AG": "Zürich",
    "Zühlke Engineering AG": "Zürich",
    "Swiss International Air Lines AG": "Zürich",
    "swissQuant Group AG": "Zürich",
    "comparis.ch AG": "Zürich",
    "SMG Swiss Marketplace Group": "Zürich",
    "AutoForm Development GmbH": "Zürich",
    "Vontobel Beteiligungen AG": "Zürich",
    "Sonova AG": "Zürich",
    "D ONE Value Creation AG": "Zürich",
    "adesso Schweiz AG": "Zürich",
    "myScience": "Zürich",
    "ServiceNow": "Zürich",
    "Deloitte": "Zürich",
    "NielsenIQ": "Zürich",
    "Octapharma AG": "Zürich",
    "Ernst & Young AG": "Zürich",
    "Swiss Life AG": "Zürich",
    "Sika AG": "Zürich",
    "ISS Schweiz": "Zürich",
    "PPCmetrics AG": "Zürich",
    "Supercomputing Systems AG": "Zürich",
    "Wingtra AG": "Zürich",
    "Frontify AG": "Zürich",
    "isolutions AG": "Zürich",
    "Impact Hub Zürich AG": "Zürich",
    "SCOR Services Switzerland AG": "Zürich",
    "Jabil Switzerland Manufacturing GmbH": "Zürich",
    "SR Technics Switzerland AG": "Zürich",
    "Edelweiss Air AG": "Zürich",
    "Zürcher Hochschule der Künste": "Zürich",
    "ZüriPharm AG": "Zürich",
    "Maag Pump Systems AG": "Zürich",
    "FIS (Switzerland) SA, Zweigniederlassung Zürich": "Zürich",
    "Medartis AG": "Zürich",
    "ZHAW Zürcher Hochschule für Angewandte Wissenschaften": "Winterthur",
    "Winterthur Gas & Diesel AG": "Winterthur",
    "EAWAG Dübendorf, Eidg. Anstalt für Wasser-, Abwasserreinigung & Gewässerschutz": "Dübendorf",
    # Genève
    "CERN European Organization for Nuclear Research": "Meyrin",
    "Banque Pictet & Cie SA": "Genève",
    "Bank Julius Bär & Co. AG": "Genève",
    "Banque Lombard Odier & Cie SA": "Genève",
    "Vitol": "Genève",
    "Kudelski S.A.": "Genève",
    "H55 SA": "Genève",
    "Energy Vault SA": "Genève",
    "HYDRO Exploitation SA": "Genève",
    "Kepler Cheuvreux (Suisse) SA": "Genève",
    "SIB Institut Suisse de Bioinformatique": "Genève",
    # Basel
    "Novartis AG": "Basel",
    "Universitätsspital Basel": "Basel",
    "Basler Kantonalbank": "Basel",
    "Bank Cler AG": "Basel",
    "Syngenta Group": "Basel",
    "Manor AG": "Basel",
    "Huntsman Advanced Materials (Switzerland) GmbH": "Münchenstein",
    # Bern
    "Hamilton Services AG": "Bern",
    "Energie Wasser Bern": "Bern",
    "Implenia Schweiz AG": "Bern",
    "Ramboll AG": "Bern",
    "CSL Behring AG": "Bern",
    "Comfone AG": "Bern",
    "RUAG AG": "Bern",
    "Mobility Genossenschaft": "Bern",
    # Lausanne / Vaud
    "Université de Lausanne": "Lausanne",
    "UCB-Pharma SA": "Lausanne",
    "Magellan Partners": "Lausanne",
    "salesforce.com Sàrl": "Lausanne",
    "Amaris Consulting Sàrl": "Lausanne",
    "Talan SA": "Lausanne",
    "selected sa": "Lausanne",
    "MANTA GROUP SA": "Lausanne",
    "VAUDOISE ASSURANCES HOLDING SA": "Lausanne",
    "Bobst Mex SA": "Mex",
    "VF International Sagl": "Stabio",
    # Aargau
    "Baloise Versicherung AG": "Liestal",
    "Mettler-Toledo GmbH": "Greifensee",
    "Thermo Fisher Scientific (Schweiz) AG": "Reinach",
    "GF Machining Solutions Management SA": "Biel/Bienne",
    "Paul Scherrer Institut (PSI)": "Villigen",
    "Peoplefone AG": "Aarau",
    "finnova AG Bankware": "Lenzburg",
    # St. Gallen
    "Stadler Rail Group": "Bussnang",
    "St.Galler Stadtwerke": "St. Gallen",
    "Büchi Labortechnik AG": "Flawil",
    "CH Media Holding AG": "St. Gallen",
    # Luzern
    "Hochschule Luzern (HSLU)": "Luzern",
    # Ticino
    "Agie Charmilles SA, Losone": "Losone",
    "Officine Ghidoni SA": "Bellinzona",
    # Fribourg
    "Etat du canton de Fribourg": "Fribourg",
    # Liechtenstein / GR
    "LGT Bank AG": "Vaduz",
    "Bank Frick AG": "Balzers",
}
    
 

def extract_city(location_raw):
    if pd.isna(location_raw):
        return None
    loc = str(location_raw).strip()

    # 1. Known company → city
    if loc in company_to_city:
        return company_to_city[loc]

    # 2. Skip company markers
    if any(m in loc for m in company_markers):
        return None

    # 3. Skip region labels
    if any(j in loc for j in region_junk):
        return None

    # 4. Extract first token before comma, strip parentheses
    city = re.sub(r'\(.*?\)', '', loc.split(",")[0]).strip()
    return city if city else None

df["city"] = df["location_raw"].apply(extract_city)

In [20]:
# Canton → Region Mapping

region_map = {
    "ZH": "Zurich",
    "BE": "Espace Mittelland",
    "LU": "Central Switzerland",
    "UR": "Central Switzerland",
    "SZ": "Central Switzerland",
    "OW": "Central Switzerland",
    "NW": "Central Switzerland",
    "GL": "Eastern Switzerland",
    "ZG": "Central Switzerland",
    "FR": "Espace Mittelland",
    "SO": "Espace Mittelland",
    "BS": "Northwestern Switzerland",
    "BL": "Northwestern Switzerland",
    "SH": "Eastern Switzerland",
    "AR": "Eastern Switzerland",
    "AI": "Eastern Switzerland",
    "SG": "Eastern Switzerland",
    "GR": "Eastern Switzerland",
    "AG": "Northwestern Switzerland",
    "TG": "Eastern Switzerland",
    "TI": "Ticino",
    "VD": "Lake Geneva Region",
    "VS": "Lake Geneva Region",
    "NE": "Espace Mittelland",
    "GE": "Lake Geneva Region",
    "JU": "Espace Mittelland"
}

df["region"] = df["canton"].map(region_map)

In [21]:
# Apply region mapping
df["region"] = df["canton"].map(region_map)
print("Region distribution:")
print(df["region"].value_counts())

Region distribution:
region
Zurich                      242
Lake Geneva Region          184
Espace Mittelland            88
Northwestern Switzerland     56
Eastern Switzerland          39
Central Switzerland          31
Ticino                        4
Name: count, dtype: int64


## Step 9: Workload Cleaning

The raw `workload_raw` field (e.g. `"80 – 100%"`) is parsed into two numeric columns: `workload_min` and `workload_max`. Single values (e.g. `"100%"`) are assigned to both columns. Rows without workload information remain `NaN`.

In [22]:
# Parse workload into numeric min/max
# "80 – 100%" → workload_min=80, workload_max=100
# "100%"       → workload_min=100, workload_max=100

import re

def parse_workload(raw):
    if pd.isna(raw):
        return None, None
    nums = re.findall(r'\d+', str(raw))
    if len(nums) == 1:
        v = int(nums[0])
        return v, v
    elif len(nums) >= 2:
        return int(nums[0]), int(nums[1])
    return None, None

df[["workload_min", "workload_max"]] = df["workload_raw"].apply(
    lambda x: pd.Series(parse_workload(x))
)

print(df[["workload_raw", "workload_min", "workload_max"]].drop_duplicates().sort_values("workload_min"))


    workload_raw  workload_min  workload_max
118     5 – 100%           5.0         100.0
208    10 – 100%          10.0         100.0
616          15%          15.0          15.0
1            20%          20.0          20.0
0       40 – 50%          40.0          50.0
351    50 – 100%          50.0         100.0
455          50%          50.0          50.0
2      60 – 100%          60.0         100.0
71           60%          60.0          60.0
195     60 – 80%          60.0          80.0
649    70 – 100%          70.0         100.0
3      80 – 100%          80.0         100.0
96           80%          80.0          80.0
261    90 – 100%          90.0         100.0
312    95 – 100%          95.0         100.0
4           100%         100.0         100.0
67           NaN           NaN           NaN


## Step 10: Sort & Final Column Summary

Columns are reordered into a logical schema grouping identifiers, location, dates, job attributes, salary, skills, and description. A final summary prints the dataset shape, dtypes, and missing value counts for each variable as a handoff check before saving.

In [23]:
# Define final column order
final_cols = [
    # identifiers
    "job_id", "title", "company", "role",
    # location
    "location_raw", "city", "canton", "region",
    # dates
    "posted_date", "quarter", "macro_quarter",
    # job attributes
    "seniority", "workload_min", "workload_max",
    # salary
    "salary_available", "salary_min", "salary_max",
    # skills
    "skills", "skill_count",
    # text
    "description",
]

# Keep only columns that exist (graceful handling if any are missing)
final_cols = [c for c in final_cols if c in df.columns]
df = df[final_cols]

print("=" * 55)
print("FINAL DATASET SUMMARY")
print("=" * 55)
print(f"Shape:        {df.shape[0]} rows × {df.shape[1]} columns")
print()
print("── Dtypes & Missing Values ───────────────────────────")
summary = pd.DataFrame({
    "dtype":   df.dtypes,
    "missing": df.isnull().sum(),
    "missing%": (df.isnull().mean() * 100).round(1)
})
print(summary.to_string())
print()
print("── Key Variable Distributions ────────────────────────")
print("\nRole:");        print(df["role"].value_counts())
print("\nSeniority:");   print(df["seniority"].value_counts(dropna=False))
print("\nRegion:");      print(df["region"].value_counts(dropna=False))
print("\nSkill count:"); print(df["skill_count"].describe().round(1))

FINAL DATASET SUMMARY
Shape:        743 rows × 20 columns

── Dtypes & Missing Values ───────────────────────────
                           dtype  missing  missing%
job_id                    object        0       0.0
title                     object        0       0.0
company                   object        0       0.0
role                      object        0       0.0
location_raw              object        0       0.0
city                      object       86      11.6
canton                    object       99      13.3
region                    object       99      13.3
posted_date       datetime64[ns]        3       0.4
quarter                   object        0       0.0
macro_quarter             object        0       0.0
seniority                 object      394      53.0
workload_min             float64       20       2.7
workload_max             float64       20       2.7
salary_available           int64        0       0.0
salary_min               float64      734      98.8
sa

## Step 11: Save Final Dataset

The cleaned dataset is saved to `data/processed/jobs_micro_cleaned_final.csv` for use in Phase 3 macro-level integration. The output directory is created automatically if it does not exist.

In [24]:
import os

output_path = "../data/processed/jobs_micro_cleaned_final.csv"

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)

df.to_csv(output_path, index=False)

print(f"Saved:   {output_path}")
print(f"Shape:   {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")

Saved:   ../data/processed/jobs_micro_cleaned_final.csv
Shape:   743 rows × 20 columns
Columns: ['job_id', 'title', 'company', 'role', 'location_raw', 'city', 'canton', 'region', 'posted_date', 'quarter', 'macro_quarter', 'seniority', 'workload_min', 'workload_max', 'salary_available', 'salary_min', 'salary_max', 'skills', 'skill_count', 'description']
